## Gold Layer

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *

In [0]:
## first time full load to be done in SAles Fact Table from SILVER------------>>GOLD

sales_df=spark.sql("select * from retailfashiondata.silvertransformed.sales_fact")
display(sales_df)

## intial load done using the full load.

# (sales_df.write.format("delta")
#               .mode("overwrite")
#               .saveAsTable("retailfashiondata.gold.sales_fact"))

In [0]:
# Get the max ingestion_time from the target (gold) table
last_modified_date = spark.sql("SELECT max(ingestion_time) FROM retailfashiondata.gold.sales_fact").collect()[0][0]


sales_fact= spark.sql("select * from retailfashiondata.silvertransformed.sales_fact")


sales_fact= sales_fact.filter(col("ingestion_time")>last_modified_date)

display(sales_fact)

(sales_fact.write.format("delta").mode("append").saveAsTable("retailfashiondata.gold.sales_fact"))




In [0]:
%sql
MERGE INTO retailfashiondata.gold.sales_fact AS t
USING sales_fact s
ON t.transaction_id = s.transaction_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
Select * from  retailfashiondata.gold.sales_fact;

In [0]:
##create table cust_dim

spark.sql("""Create Table if not exists retailfashiondata.gold.cust_dim(
customer_id string,
age integer,
gender string,
city string,
email string,
ingestion_time timestamp,
start_date date,
end_date date,
is_current string)
          Using DELTA""")


# create table prod_dim

spark.sql("""Create Table if not exists retailfashiondata.gold.prod_dim(
product_id string,
category string,
color string,
size string,
season string,
supplier string,
cost_price double,
list_price double,
ingestion_time timestamp,
profit_margin double,
start_date date,
end_date date,
is_current string)
          Using DELTA""")

# create store dim

spark.sql("""Create Table if not exists retailfashiondata.gold.store_dim(
store_id string,
store_name string,
region string,
store_size_m2 integer,
ingestion_time timestamp,
start_date date,
end_date date,
is_current string)
          Using DELTA""")


# create sales fact

spark.sql("""Create Table if not exists retailfashiondata.gold.sales_fact(
transaction_id string,
date date,
product_id string,
store_id string,
customer_id string,
quantity integer,
discount double,
returned boolean,
ingestion_time timestamp
)USING DELTA""")



In [0]:


target = DeltaTable.forName(spark,"retailfashiondata.gold.cust_dim")
source = spark.table("retailfashiondata.silvertransformed.cust_dim")

target.alias("t").merge(
    source.alias("s"),"t.customer_id=s.customer_id"
).whenMatchedUpdate(condition= "t.is_current='True' and(t.city <> s.city OR t.email <> s.email)",
    set={
        "end_date" : lit(current_date()),
         "is_current" : lit(False)
    }
).whenNotMatchedInsert(
    values={
        "customer_id" : "s.customer_id",
        "age" :"s.age",
        "gender" :"s.gender",
        "city" : "s.city",
        "email" : "s.email",
        "ingestion_time" : "s.ingestion_time",
        "start_date": lit(current_date()),
        "end_date" : lit(None),
        "is_current" : lit(True)
    }
).whenNotMatchedBySourceUpdate(condition="is_current='True'",
                               set={
                                   "end_date" : lit(current_date()),
                                   "is_current" : lit(False)
}).execute()



In [0]:
%sql
Select * from retailfashiondata.gold.cust_dim;

In [0]:


target = DeltaTable.forName(spark,"retailfashiondata.gold.prod_dim")
source = spark.table("retailfashiondata.silvertransformed.prod_dim")

target.alias("t").merge(
    source.alias("s"),"t.product_id=s.product_id"
).whenMatchedUpdate(condition= "t.is_current='True' and(t.category <> s.category OR t.color <> s.color OR t.size <> s.size OR t.season <> s.season OR t.supplier <> s.supplier OR t.cost_price <> s.cost_price OR t.list_price <> s.list_price)",
    set={
        "end_date" : lit(current_date()),
         "is_current" : lit(False)
    }
).whenNotMatchedInsert(
    values={
        "product_id" : "s.product_id" ,
        "category" : "s.category",
        "color" : "s.color",
        "size" : "s.size",
        "season" : "s.season",
        "supplier" : "s.supplier",
        "cost_price" : "s.cost_price",
        "list_price" : "s.list_price",
        "ingestion_time" :"s.ingestion_time",
        "profit_margin" : "s.profit_margin",
        "start_date": lit(current_date()),
        "end_date" : lit(None),
        "is_current" : lit(True)
    }
).whenNotMatchedBySourceUpdate(condition="is_current='True'",
                               set={
                                   "end_date" : lit(current_date()),
                                   "is_current" : lit(False)
}).execute()



In [0]:
%sql
select * from retailfashiondata.gold.prod_dim;

In [0]:



target = DeltaTable.forName(spark,"retailfashiondata.gold.store_dim")
source = spark.table("retailfashiondata.silvertransformed.store_dim")

target.alias("t").merge(
    source.alias("s"),"t.store_id=s.store_id"
).whenMatchedUpdate(condition= "t.is_current='True' and(t.region <> s.region OR t.store_name <> s.store_name OR t.store_size_m2 <> s.store_size_m2)",
    set={
        "end_date" : lit(current_date()),
         "is_current" : lit(False)
    }
).whenNotMatchedInsert(
    values={
        "store_id" : "s.store_id",
        "store_name" : "s.store_name",
        "region" : "s.region",
        "store_size_m2": "s.store_size_m2",
        "ingestion_time" : "s.ingestion_time",
        "start_date": lit(current_date()),
        "end_date" : lit(None),
        "is_current" : lit(True)
    }
).whenNotMatchedBySourceUpdate(condition="is_current='True'",
                               set={
                                   "end_date" : lit(current_date()),
                                   "is_current" : lit(False)
}).execute()



In [0]:
%sql
select * from retailfashiondata.gold.store_dim;